# 01 — Vehicle Detection on LVO Satellite Imagery

**Objective:** Count military vehicles visible at Russian bases in the Leningrad Military District using YOLOv8 on commercial satellite imagery (50cm SkySat, 35cm WorldView-3).

**Method:** Tile → YOLO → NMS dedup → count → compare with claimed numbers

**Imagery:** [SkyFi](https://skyfi.com) | **Unit tracking:** [ISW](https://www.understandingwar.org/) | **Analysis:** [EstWarden](https://estwarden.eu)

In [ ]:
from PIL import Image as _PILImage
_PILImage.MAX_IMAGE_PIXELS = None  # Allow large satellite images

import os, json, numpy as np, cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from pathlib import Path
from ultralytics import YOLO
from collections import Counter

DATA_DIR = Path("../data/fullres")
OUTPUT_DIR = Path("../outputs/01-vehicle-detection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for f in sorted(DATA_DIR.glob("*")):
    img = Image.open(f)
    print(f"{f.name:55} {img.size[0]:>6}x{img.size[1]:<6} {os.path.getsize(f)/1e6:>7.1f}MB {img.mode}")

## Step 1: Tile images into 640×640 patches with overlap

In [ ]:
def tile_image(image_path, tile_size=640, overlap=64):
    img = cv2.imread(str(image_path))
    h, w = img.shape[:2]
    stride = tile_size - overlap
    tiles = []
    for y in range(0, h - tile_size + 1, stride):
        for x in range(0, w - tile_size + 1, stride):
            tiles.append({"image": img[y:y+tile_size, x:x+tile_size],
                         "x_offset": x, "y_offset": y})
    print(f"Image: {w}x{h} → {len(tiles)} tiles ({tile_size}px, overlap={overlap}px)")
    return tiles, (h, w)

pskov_skysat = DATA_DIR / "pskov-76vdv-skysat-2026-03-14-fullres.png"
tiles_pskov, shape_pskov = tile_image(pskov_skysat)

## Step 2: Run YOLOv8 vehicle detection on all tiles

In [ ]:
model = YOLO("yolov8x.pt")  # downloads ~130MB first time
VEHICLE_CLASSES = {2: "car", 5: "bus", 7: "truck"}

def detect_vehicles(tiles, model, conf=0.25):
    dets = []
    for i, t in enumerate(tiles):
        if i % 100 == 0: print(f"  tile {i}/{len(tiles)}...")
        for r in model.predict(t["image"], conf=conf, verbose=False, device="cuda"):
            for box in r.boxes:
                cid = int(box.cls[0])
                if cid in VEHICLE_CLASSES:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    dets.append({"class": VEHICLE_CLASSES[cid], "conf": float(box.conf[0]),
                                "x1": int(x1+t["x_offset"]), "y1": int(y1+t["y_offset"]),
                                "x2": int(x2+t["x_offset"]), "y2": int(y2+t["y_offset"])})
    return dets

print("Running YOLO on Pskov SkySat...")
det_pskov_raw = detect_vehicles(tiles_pskov, model)
print(f"Raw detections: {len(det_pskov_raw)}")

## Step 3: Global NMS to remove duplicate detections from overlapping tiles

In [ ]:
def global_nms(dets, iou_thresh=0.5):
    if not dets: return []
    boxes = np.array([[d["x1"],d["y1"],d["x2"],d["y2"]] for d in dets], dtype=float)
    scores = np.array([d["conf"] for d in dets])
    order = scores.argsort()[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]; keep.append(i)
        if len(order) == 1: break
        xx1 = np.maximum(boxes[i,0], boxes[order[1:],0])
        yy1 = np.maximum(boxes[i,1], boxes[order[1:],1])
        xx2 = np.minimum(boxes[i,2], boxes[order[1:],2])
        yy2 = np.minimum(boxes[i,3], boxes[order[1:],3])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        a_i = (boxes[i,2]-boxes[i,0])*(boxes[i,3]-boxes[i,1])
        a_j = (boxes[order[1:],2]-boxes[order[1:],0])*(boxes[order[1:],3]-boxes[order[1:],1])
        iou = inter / (a_i + a_j - inter + 1e-6)
        order = order[np.where(iou <= iou_thresh)[0] + 1]
    return [dets[i] for i in keep]

det_pskov = global_nms(det_pskov_raw)
print(f"After NMS: {len(det_pskov)} unique vehicles")
print(f"By class: {Counter(d['class'] for d in det_pskov)}")
print(f"High confidence (>0.5): {len([d for d in det_pskov if d['conf']>0.5])}")

## Step 4: Visualize detections

In [ ]:
def plot_detections(img_path, dets, title, out_path, max_dim=4096):
    img = cv2.imread(str(img_path)); h, w = img.shape[:2]
    s = min(max_dim/w, max_dim/h, 1.0)
    if s < 1: img = cv2.resize(img, (int(w*s), int(h*s)))
    fig, ax = plt.subplots(1, 1, figsize=(20, 16))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    colors = {"car":"cyan", "bus":"yellow", "truck":"red"}
    for d in dets:
        ax.add_patch(Rectangle((d["x1"]*s, d["y1"]*s), (d["x2"]-d["x1"])*s, (d["y2"]-d["y1"])*s,
                               lw=1, ec=colors.get(d["class"],"white"), fc="none"))
    c = Counter(d["class"] for d in dets)
    ax.set_title(f"{title}\nDetected: {len(dets)} ({', '.join(f'{k}:{v}' for k,v in c.items())})", fontsize=14)
    ax.axis("off"); plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight"); plt.show()

plot_detections(pskov_skysat, det_pskov, "Pskov 76th VDV — SkySat 50cm — 2026-03-14",
               OUTPUT_DIR / "pskov_76vdv_detections.png")

## Step 5: Repeat for Pskov Cherekha (WorldView-3 35cm)

In [ ]:
pskov_wv3 = DATA_DIR / "pskov-cherekha-wv3-2026-02-05-fullres.png"
tiles_wv3, shape_wv3 = tile_image(pskov_wv3)
print("Running YOLO on Pskov Cherekha WV3...")
det_wv3_raw = detect_vehicles(tiles_wv3, model)
det_wv3 = global_nms(det_wv3_raw)
print(f"Cherekha: {len(det_wv3)} vehicles (raw: {len(det_wv3_raw)})")
plot_detections(pskov_wv3, det_wv3, "Pskov Cherekha — WV3 35cm — 2026-02-05",
               OUTPUT_DIR / "pskov_cherekha_detections.png")

## Step 6: Summary — detected vs. claimed

In [ ]:
total = len(det_pskov) + len(det_wv3)
hc = len([d for d in det_pskov+det_wv3 if d["conf"]>0.5])
results = {
    "pskov_76vdv": {"sensor":"SkySat 50cm","date":"2026-03-14","count":len(det_pskov),
                    "high_conf":len([d for d in det_pskov if d["conf"]>0.5]),
                    "by_class":dict(Counter(d["class"] for d in det_pskov))},
    "pskov_cherekha": {"sensor":"WV3 35cm","date":"2026-02-05","count":len(det_wv3),
                       "high_conf":len([d for d in det_wv3 if d["conf"]>0.5]),
                       "by_class":dict(Counter(d["class"] for d in det_wv3))}
}
print("="*60)
print("LVO FORCE POSTURE — VEHICLE DETECTION SUMMARY")
print("="*60)
for n, r in results.items():
    print(f"  {n}: {r['count']} vehicles ({r['high_conf']} high-conf) — {r['sensor']} {r['date']}")
print(f"\n  TOTAL: {total} ({hc} high-confidence)")
print(f"\n  Note: COCO-pretrained YOLO = UPPER BOUND (includes civilian).")
print(f"  True military vehicle count is LOWER.")
with open(OUTPUT_DIR / "results.json", "w") as f: json.dump(results, f, indent=2)
print(f"\nSaved: {OUTPUT_DIR/'results.json'}")

## Limitations & Next Steps

**Limitations:**
1. YOLO trained on COCO (consumer photos), not satellite — [xView](https://xviewdataset.org/)/[DOTA](https://captain-whu.github.io/DOTA/) would be better
2. 50cm = tank is ~14×7 pixels — detectable but not classifiable
3. Preview images have compression vs full GeoTIFF
4. Civilian vehicles inflate count
5. Camouflage/hangars invisible to optical

**Next notebooks:**
- `02-super-resolution.ipynb` — Real-ESRGAN upscaling before detection
- `03-satellite-finetune.ipynb` — Fine-tune YOLO on xView/DIOR satellite data
- `04-sar-analysis.ipynb` — ICEYE SAR radar vehicle detection
- `05-temporal-comparison.ipynb` — Feb vs Mar change detection